# Clase 3 — Histogramas y gráficos de barras

En la clase anterior contamos historias **a lo largo del tiempo** (líneas, áreas). Hoy respondemos dos preguntas distintas:

- ¿Cómo se **distribuyen** los valores? → **Histograma**
- ¿Cómo se **comparan** categorías con nombre entre sí? → **Gráfico de barras**

Seguimos trabajando con `df_can`, el dataset de inmigración a Canadá (1980-2013).

## 0. Preparación: librerías y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
mpl.style.use('ggplot')

print('Matplotlib version:', mpl.__version__)

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Datos cargados correctamente')
df_can.head()

In [ ]:
df_can.set_index('Country', inplace=True)

# lista de años 1980-2013 como texto, la usaremos todo el curso
years = list(map(str, range(1980, 2014)))

print(df_can.shape)
df_can.head()

## 1. Histogramas: organizar la multitud en grupos

**Analogía:** imagina un estadio lleno de gente y quieres saber cómo se distribuyen las edades. No preguntas la edad exacta de cada persona; las agrupas en rangos (0-10, 11-20, 21-30…) y cuentas cuánta gente cae en cada rango. Eso es un histograma: un conjunto de **bins** (cajones) con la cantidad de elementos que caen dentro de cada uno.

Un histograma **no** compara categorías con nombre (Canadá vs México). Compara **rangos de un valor numérico**, para revelar dónde se concentra la mayoría de los datos y dónde hay valores extremos.

### 1.1 Histograma simple con `np.histogram`

In [ ]:
# np.histogram devuelve 2 valores: el conteo por bin y los límites de cada bin
count, bin_edges = np.histogram(df_can['2013'])

print(count)      # cuántos países caen en cada bin
print(bin_edges)  # los límites de cada bin (por defecto, 10 bins)

Con esto leemos la distribución en texto plano: por ejemplo, "178 países aportaron entre 0 y 3412.9 inmigrantes en 2013, 11 países aportaron entre 3412.9 y 6825.8", y así sucesivamente. Unos pocos países (India, China) concentran cifras enormes frente a la mayoría.

In [ ]:
df_can['2013'].plot(kind='hist', figsize=(8, 5))

plt.title('Histograma de inmigración desde 195 países en 2013')
plt.ylabel('Número de países')
plt.xlabel('Número de inmigrantes')

plt.show()

Detalle molesto: los ticks del eje x no coinciden con los límites reales de los bins. Se arregla pasando `bin_edges` como argumento `xticks`:

In [ ]:
count, bin_edges = np.histogram(df_can['2013'])

df_can['2013'].plot(kind='hist', figsize=(8, 5), xticks=bin_edges)

plt.title('Histograma de inmigración desde 195 países en 2013')
plt.ylabel('Número de países')
plt.xlabel('Número de inmigrantes')

plt.show()

### 1.2 Histograma multi-serie: comparar varias distribuciones a la vez

¿Qué pasa si queremos comparar la distribución de inmigrantes de **Dinamarca, Noruega y Suecia** entre 1980 y 2013? El primer intento suele fallar:

```python
df_can.loc[['Denmark', 'Norway', 'Sweden'], years].plot.hist()
```

*pandas* interpreta cada **columna** del dataframe como una serie a graficar, y aquí las columnas son los **años**, no los países. El resultado es un histograma de años, sin sentido. La solución: **transponer** el dataframe para que los países queden como columnas y los años como filas.

In [ ]:
# transponer para que cada país sea una columna
df_t = df_can.loc[['Denmark', 'Norway', 'Sweden'], years].transpose()

df_t.plot(kind='hist', figsize=(10, 6))

plt.title('Histograma de inmigración desde Dinamarca, Noruega y Suecia (1980-2013)')
plt.ylabel('Número de años')
plt.xlabel('Número de inmigrantes')

plt.show()

Ahora afinamos: más bins para más detalle, transparencia para ver superposiciones, colores propios para cada serie.

In [ ]:
count, bin_edges = np.histogram(df_t, 15)

df_t.plot(kind='hist',
          figsize=(10, 6),
          bins=15,
          alpha=0.6,                                    # transparencia: 0 = invisible, 1 = opaco
          xticks=bin_edges,
          color=['coral', 'darkslateblue', 'mediumseagreen'])

plt.title('Histograma de inmigración desde Dinamarca, Noruega y Suecia (1980-2013)')
plt.ylabel('Número de años')
plt.xlabel('Número de inmigrantes')

plt.show()

Si preferimos que las barras no se solapen visualmente, las apilamos con `stacked=True`:

In [ ]:
count, bin_edges = np.histogram(df_t, 15)
xmin = bin_edges[0] - 10   # margen estético a la izquierda
xmax = bin_edges[-1] + 10  # margen estético a la derecha

df_t.plot(kind='hist',
          figsize=(10, 6),
          bins=15,
          xticks=bin_edges,
          color=['coral', 'darkslateblue', 'mediumseagreen'],
          stacked=True,
          xlim=(xmin, xmax))

plt.title('Histograma de inmigración desde Dinamarca, Noruega y Suecia (1980-2013)')
plt.ylabel('Número de años')
plt.xlabel('Número de inmigrantes')

plt.show()

## 2. Gráficos de barras: comparar categorías cara a cara

Un gráfico de barras no reparte valores en rangos: compara **categorías con nombre**. La longitud de cada barra representa la magnitud de esa categoría. Piensa en una tabla de clasificación deportiva: cada equipo es una barra, y la altura de la barra es su puntuación. De un vistazo ves quién va primero y quién va último.

En *pandas*, `kind='bar'` dibuja barras **verticales** y `kind='barh'` dibuja barras **horizontales**.

### 2.1 Barras verticales: el caso de Islandia

Entre 2008 y 2011, Islandia sufrió el mayor colapso bancario relativo de la historia económica moderna. Veamos si eso se refleja en la inmigración islandesa a Canadá.

In [ ]:
df_iceland = df_can.loc['Iceland', years]

df_iceland.plot(kind='bar', figsize=(10, 6), rot=90)  # rot=90 rota las etiquetas del eje x

plt.xlabel('Año')
plt.ylabel('Número de inmigrantes')
plt.title('Inmigrantes islandeses a Canadá (1980-2013)')

plt.show()

El salto después de 2008 es evidente. Podemos señalarlo con una flecha usando `plt.annotate`:

In [ ]:
df_iceland.plot(kind='bar', figsize=(10, 6), rot=90)

plt.xlabel('Año')
plt.ylabel('Número de inmigrantes')
plt.title('Inmigrantes islandeses a Canadá (1980-2013)')

# la flecha: punta en xy, base en xytext
plt.annotate('',
             xy=(32, 70),        # punta de la flecha: año 2012, población 70
             xytext=(28, 20),    # base de la flecha: año 2008, población 20
             xycoords='data',    # usa el sistema de coordenadas del propio gráfico
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))

# el texto que acompaña a la flecha
plt.annotate('Crisis financiera 2008-2011',
             xy=(28, 30),
             rotation=72.5,   # ángulo ajustado a ojo para que siga la flecha
             va='bottom',     # alineación vertical del texto
             ha='left')       # alineación horizontal del texto

plt.show()

`xy` es el punto de destino (la punta de la flecha), `xytext` es el punto de origen (donde empieza el texto o la cola de la flecha), y `arrowprops` controla el estilo del trazo. No hace falta memorizar cada parámetro: se ajustan por prueba y error hasta que el resultado se ve bien.

### 2.2 Barras horizontales: cuando el nombre no cabe

Cuando las etiquetas son largas (nombres de países), las barras verticales las aplastan unas contra otras. La solución: girar el gráfico 90 grados con `kind='barh'`, que deja mucho más espacio en el eje vertical para el texto.

Mostremos los 15 países que más inmigrantes aportaron a Canadá entre 1980 y 2013, con el valor total anotado al final de cada barra.

In [ ]:
# paso 1: ordenar y quedarnos con los 15 mayores
df_can.sort_values(by='Total', ascending=True, inplace=True)
df_top15 = df_can['Total'].tail(15)

# paso 2: graficar
df_top15.plot(kind='barh', figsize=(12, 12), color='steelblue')
plt.xlabel('Número de inmigrantes')
plt.title('Los 15 países que más contribuyeron a la inmigración a Canadá (1980-2013)')

# paso 3: anotar el valor de cada barra
# enumerate() nos da el índice (posición vertical) y el valor de cada país a la vez
for index, value in enumerate(df_top15):
    label = format(int(value), ',')  # formatea el número con comas de miles
    # colocamos el texto cerca del final de la barra, ajustando manualmente el desplazamiento
    plt.annotate(label, xy=(value - 47000, index - 0.10), color='white')

plt.show()

El bucle `enumerate(df_top15)` es la pieza clave: recorre los valores de la serie y, a la vez, te da la posición (`index`) de cada uno, que corresponde a la altura de cada barra horizontal. Sin `enumerate` tendrías que llevar la cuenta de la posición a mano.

## Puntos clave

- El **histograma** reparte un valor numérico en bins y cuenta cuántos elementos caen en cada uno; responde "¿cómo se distribuyen mis datos?".
- El **gráfico de barras** compara categorías con nombre; responde "¿quién tiene más y quién tiene menos?".
- Si un histograma de un dataframe sale mal, sospecha primero de la orientación de los datos: probablemente necesitas transponer con `.transpose()`.
- `np.histogram()` te da los bins y conteos en crudo antes de dibujar nada; úsalo para pasar `bin_edges` como `xticks`.
- `kind='bar'` es vertical, `kind='barh'` es horizontal; usa barras horizontales cuando las etiquetas de categoría son largas.
- `enumerate()` dentro de un bucle te da índice y valor a la vez, ideal para anotar cada barra con `plt.annotate`.

## Siguiente paso

Abre `clase03_practica_histogramas-barras.ipynb` y resuelve los ejercicios propuestos.